# Chapter 5 — Claude Code: Configuration and Workflows · Homework (self-test)

**Goal:** check that you can place Claude Code configuration at the right scope, write skills and rules that load when they should, and run Claude Code safely in CI.

- **Theory:** [README.md](README.md) · **Practice:** [01_practice.ipynb](01_practice.ipynb)
- **Estimated time:** 2–2.5 hours
- **Estimated API cost:** **$0**. Every exercise is checked offline; nothing here calls Claude.
- **Prerequisites:** the repo venv from [00 — Prerequisites](../../00-prerequisites/README.md). Work through the practice notebook first.

### How to use this notebook

1. Answer the quiz and solve the exercises **without scrolling to the bottom**. Solutions and explanations are in the last section, inside collapsed blocks.
2. Run each `check_exN()` cell after you edit an exercise. Checkers never crash the notebook; they print a hint instead.
3. Write your Part C answer in the markdown cell provided, then compare it with the rubric.
4. Run the **Scorecard** cell.

**Scoring:** quiz score (%) and exercises passed (%) are averaged. The pass mark is **72%**, which mirrors the Claude certification exam's 720/1000.

## Setup

In [ ]:
import hashlib
import json
import os
import re

import yaml  # PyYAML ships with JupyterLab
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")

RESULTS: dict[str, object] = {}   # filled by grade_quiz() and the check_exN() cells


def parse_frontmatter(text: str) -> tuple[dict, str]:
    """Split `---` YAML frontmatter from the body (same helper as in the practice notebook)."""
    m = re.match(r"\A---[ \t]*\n(.*?)\n---[ \t]*(?:\n|\Z)", text, flags=re.S)
    if not m:
        return {}, text
    try:
        meta = yaml.safe_load(m.group(1)) or {}
    except yaml.YAMLError:
        return {}, text
    return (meta if isinstance(meta, dict) else {}), text[m.end():]


print("model:", MODEL)

In [ ]:
# No exercise in this homework calls Claude, so no key is required.
print("ANTHROPIC_API_KEY set" if os.getenv("ANTHROPIC_API_KEY") else
      "ANTHROPIC_API_KEY not set (fine: everything here is offline)")

## Part A — Quiz

Twelve scenario questions. Each has exactly one best answer.

**Q1** *(concept 5.1)*. You want Claude Code to know that **your** local sandbox API runs on `http://localhost:8081` when you work in the `acme-shop` repo. Your teammates use different ports. Where does this instruction go?

- **A)** `./CLAUDE.md` in the repo, under a "Local setup" heading
- **B)** `~/.claude/CLAUDE.md`
- **C)** `./CLAUDE.local.md`, listed in `.gitignore`
- **D)** `.claude/rules/local.md` with `paths: "**/*"`

**Q2** *(concept 5.2)*. A team splits its 600-line CLAUDE.md into six topic files and replaces the content with six `@docs/...` imports, expecting every session to use fewer tokens. What actually happens?

- **A)** Token use drops, because imported files load only when Claude decides they are relevant
- **B)** Token use stays about the same, because imported files are expanded into context at launch
- **C)** Imports beyond the first three are ignored, so token use drops but some rules are lost
- **D)** The imports fail, because CLAUDE.md can only import files inside `.claude/`

**Q3** *(concept 5.3)*. `.claude/rules/sql.md` has `paths: "**/*.sql"` in its frontmatter. You ask Claude to explain the overall architecture, and it reads only `README.md` and `docs/overview.md`. Is the SQL rule in Claude's context?

- **A)** Yes: every file in `.claude/rules/` loads at launch
- **B)** Only its frontmatter is loaded, so Claude knows the rule exists
- **C)** Only if the developer runs `/rules sql` first
- **D)** No: a path-scoped rule loads only when Claude reads, writes or edits a file that matches its glob

**Q4** *(concept 5.5)*. Claude ran your team's `/deploy-prod` skill on its own after a teammate typed "the staging deploy looks good". You want only humans to be able to trigger it. What do you change?

- **A)** Add `disable-model-invocation: true` to the skill's frontmatter
- **B)** Add `user-invocable: false` to the skill's frontmatter
- **C)** Append "Never run automatically" to the skill's description
- **D)** Add `allowed-tools: Bash(./deploy.sh)` so only the deploy script can run

**Q5** *(concept 5.5)*. Security requires that while your `/summarize-logs` skill is active, Claude has no access to the Bash, Write or Edit tools at all. According to the current Claude Code docs, which frontmatter field does that?

- **A)** `allowed-tools: Read Grep`
- **B)** `context: fork`
- **C)** `description: "Read-only log summarizer"`
- **D)** `disallowed-tools: Bash Write Edit`

**Q6** *(concept 5.9)*. A nightly `claude -p` job behaves differently on two runners. On one of them, a hook from a teammate's `~/.claude/settings.json` fires, and an MCP server from the repo's `.mcp.json` connects. You want identical behavior on every runner. What is the best fix?

- **A)** Add `--permission-mode plan`
- **B)** Add `--no-session-persistence`
- **C)** Add `--bare`, and pass the context the job needs explicitly (for example `--append-system-prompt-file CLAUDE.md` and `--mcp-config`)
- **D)** Add `--max-turns 1`

**Q7** *(concept 5.6)*. You must replace the logging library in about 60 files. Two designs are viable (an adapter shim, or rewriting every call site), and you have never worked in this codebase. What do you do first?

- **A)** Use plan mode: let Claude explore and propose an approach, review the plan, then execute it
- **B)** Use direct execution with very detailed instructions written up front
- **C)** Use direct execution, and switch to plan mode if it gets messy
- **D)** Run `/compact` to make room in the context, then start editing

**Q8** *(concept 5.10)*. Yesterday a script ran `claude -p "Map the auth module" --output-format json`. Today a developer runs interactive `claude --continue` in the same repo and lands in an unrelated conversation. How do they get back to the scripted session?

- **A)** Run `claude --continue --fork-session`
- **B)** Run `/compact` and ask again
- **C)** They can't: `-p` sessions are never saved to disk
- **D)** Read `session_id` from the script's JSON output and run `claude --resume <session_id>`

**Q9** *(concept 5.10)*. After a long shared investigation, you want to develop two refactoring strategies in parallel, and both will edit code. Which setup keeps the two strategies from interfering?

- **A)** Resume the same session in two terminals without forking
- **B)** Fork the session twice (`--resume <id> --fork-session`) and run each fork in its own git worktree
- **C)** Fork the session twice in the same working directory, because each fork gets its own copy of the files
- **D)** Run `/clear`, paste a summary into one session, and discuss both strategies there

**Q10** *(concept 5.7)*. Two hours into a session you run `/compact`. Which of these is **re-injected from disk** afterwards, instead of surviving only as far as the summary kept it?

- **A)** A path-scoped rule that loaded when Claude read a matching test file
- **B)** A subdirectory `packages/api/CLAUDE.md` that loaded lazily
- **C)** The project-root `CLAUDE.md`
- **D)** The exact error code you typed into the chat an hour ago

**Q11** *(concept 5.11)*. Three times you asked Claude Code to "convert the legacy phone numbers to a standard format", and each attempt formats them differently. What is the most effective next instruction?

- **A)** Give 2–3 concrete input → output pairs (including one with an extension), ask Claude to add them as tests, and have it iterate until they pass
- **B)** Repeat the request and add "be consistent this time"
- **C)** Add "always format phone numbers properly" to the project CLAUDE.md
- **D)** Run `/compact` so Claude forgets the earlier attempts, then ask again in the same words

**Q12** *(concept 5.12)*. An internal dashboard must show a headless Claude Code run's text **as it is generated**, and afterwards record the run's cost and session ID. Which invocation fits?

- **A)** `claude -p "..." --output-format json`, then print `result` when the process exits
- **B)** `claude "..." --verbose`, scraping the terminal output
- **C)** `claude -p "..." --output-format text | tee run.log`
- **D)** `claude -p "..." --output-format stream-json --verbose --include-partial-messages`: print `text_delta` events as they arrive, and read cost and session ID from the final `result` line

In [ ]:
# Fill in one letter per question ("A"-"D"), then run this cell and the next.
answers = {
    "q1": None, "q2": None, "q3": None, "q4": None, "q5": None,
    "q6": None, "q7": None, "q8": None, "q9": None, "q10": None,
    "q11": None, "q12": None,
}

In [ ]:
_ANSWER_KEY = {  # sha256 of "qid:LETTER", so the answers are not readable here
    "q1": "227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e",
    "q2": "61e5302385b2dd8204dbddfd86b357cacb0de07dbc800460aa335d9bc96b6ca1",
    "q3": "14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6",
    "q4": "0eca085218afe99ae6fb326d50c6e730cd71c102ccfe2f9a10d8e6054e2018e0",
    "q5": "aa2a5c0e05888822a98d25f809f1adea4dcc57b38fe82026bca63c99ab417457",
    "q6": "18cb69e7d9c041e4c1b21fa8c0e110716664b55bef1348d49301f8891c82cef1",
    "q7": "8722338db4eb12eaae5f6f26c68875af2b115581d770f01f98b9e0e1e6010ac1",
    "q8": "85b91a1895880cedf03bb98aa9ebfb3c730c838ec5d8f59ec2d1d86bcb936ced",
    "q9": "b25dce13077091b19d36fd594efa7fa456f167c4a463522f004183e5d5d2bb5e",
    "q10": "d6ea6e76ddfa07a3b78a8bd000db76a223a8ccc3f38e290cf5a68d95e5494cc5",
    "q11": "acb9b8f6dd2a3e57b9476b90caf32ab8ad47a11c4aea20d95df7aaad0d81238c",
    "q12": "33d12c43f8f028fb844551369083e2b9daa804b371ce6b8835f63a07a5cac42e",
}


def grade_quiz(answers: dict) -> int:
    score = 0
    for qid, expected in _ANSWER_KEY.items():
        given = str(answers.get(qid) or "").strip().upper()
        ok = bool(given) and hashlib.sha256(f"{qid}:{given}".encode()).hexdigest() == expected
        score += ok
        print(f"{'✓' if ok else '✗'} {qid}: {given or '(no answer)'}")
    print(f"\nQuiz: {score}/{len(_ANSWER_KEY)} ({100 * score / len(_ANSWER_KEY):.0f}%)")
    RESULTS["quiz"] = (score, len(_ANSWER_KEY))
    return score


grade_quiz(answers)

## Part B — Coding exercises

Seven offline exercises. Edit the `TODO` cell, then run the `check_exN()` cell under it.

### Exercise 1 — Write a valid SKILL.md (concept 5.5)

Return the full text of `.claude/skills/db-migrate/SKILL.md`, with frontmatter and body. Requirements:

- frontmatter is valid YAML between `---` lines, with `name: db-migrate`
- `description` is at least 40 characters, mentions "migration", and says **when** to use the skill (include the words "Use when")
- `argument-hint` is set, for example `"[migration-name]"`
- the skill has side effects, so only a human may invoke it
- it runs in an isolated subagent context
- `WebFetch` and `WebSearch` are **removed** while it runs
- `Bash(cat db/VERSION)` is **pre-approved**
- the body uses `$ARGUMENTS` and injects the current schema version with `` !`cat db/VERSION` ``

In [ ]:
def write_db_migrate_skill() -> str:
    # TODO: return the complete SKILL.md text
    raise NotImplementedError

In [ ]:
def _split_tools(value) -> list[str]:
    if isinstance(value, list):
        return [str(v).strip() for v in value]
    return re.findall(r"[^\s,()]+(?:\([^)]*\))?", str(value or ""))


def check_ex1() -> bool:
    try:
        text = write_db_migrate_skill()
        assert isinstance(text, str) and text.startswith("---"), "the file must start with a '---' frontmatter block"
        meta, body = parse_frontmatter(text)
        assert meta, "the frontmatter did not parse as a YAML mapping"
        assert meta.get("name") == "db-migrate", "name must be exactly 'db-migrate'"
        desc = str(meta.get("description", ""))
        assert len(desc) >= 40 and "migration" in desc.lower(), "description: >= 40 chars and mention 'migration'"
        assert "use when" in desc.lower(), "description should say when to use it ('Use when ...')"
        assert meta.get("argument-hint"), "add an argument-hint"
        assert meta.get("disable-model-invocation") is True, "side effects -> only humans may invoke it (a boolean field)"
        assert meta.get("context") == "fork", "run it in an isolated subagent context"
        removed = _split_tools(meta.get("disallowed-tools"))
        assert {"WebFetch", "WebSearch"} <= set(removed), "which field REMOVES tools? (allowed-tools only pre-approves)"
        assert "Bash(cat db/VERSION)" in _split_tools(meta.get("allowed-tools")), "pre-approve Bash(cat db/VERSION)"
        assert "$ARGUMENTS" in body, "use $ARGUMENTS in the body"
        assert re.search(r"(?<!\S)!`cat db/VERSION`", body), "inject the version with !`cat db/VERSION` in the body"
    except NotImplementedError:
        print("✗ Exercise 1: not implemented yet")
        RESULTS["ex1"] = False
        return False
    except AssertionError as e:
        print(f"✗ Exercise 1: {e}")
        RESULTS["ex1"] = False
        return False
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 1: {type(e).__name__}: {e}")
        RESULTS["ex1"] = False
        return False
    print("✓ Exercise 1 passed")
    RESULTS["ex1"] = True
    return True


check_ex1()

### Exercise 2 — Where does it go? (concepts 5.1, 5.3, 5.5)

For each scenario, pick the **one** best location key from `OPTIONS`, and put it in `PLACEMENT`.

| # | Scenario |
|---|---|
| s1 | Every engineer, including new hires, must get "run `make test` before proposing a commit". |
| s2 | You like answers in British English in all of your repos. Teammates don't care. |
| s3 | Security requires a compliance reminder on every developer machine, which individual developers must not be able to exclude. |
| s4 | Conventions for every `*.tf` file, spread across 9 directories. |
| s5 | A 40-step release runbook that matters only when someone cuts a release. |
| s6 | Claude must never be able to run `terraform destroy`. |
| s7 | The self-contained `packages/billing/` package has its own money-handling conventions. |
| s8 | "Use my personal `qa-eldan` test tenant" when **you** work in this repo. |

In [ ]:
OPTIONS = {
    "managed":     "managed policy CLAUDE.md (/Library/Application Support/ClaudeCode/CLAUDE.md, /etc/claude-code/CLAUDE.md, ...)",
    "user":        "~/.claude/CLAUDE.md",
    "project":     "./CLAUDE.md or ./.claude/CLAUDE.md",
    "local":       "./CLAUDE.local.md (gitignored)",
    "subdir":      "<directory>/CLAUDE.md",
    "path_rule":   ".claude/rules/<topic>.md with paths: globs",
    "skill":       ".claude/skills/<name>/SKILL.md",
    "permissions": "permissions.deny rule or PreToolUse hook in settings",
}

# TODO: replace each None with a key from OPTIONS
PLACEMENT = {"s1": None, "s2": None, "s3": None, "s4": None,
             "s5": None, "s6": None, "s7": None, "s8": None}

In [ ]:
_EX2_HINTS = {
    "s1": "must reach everyone via git clone",
    "s2": "personal, but in every project",
    "s3": "organization-wide and cannot be excluded",
    "s4": "a file TYPE scattered across directories",
    "s5": "needed only for a TASK",
    "s6": "must be GUARANTEED, not just suggested",
    "s7": "one self-contained directory",
    "s8": "personal, this project only, never committed",
}
_EX2_EXPECTED = {  # sha256 of "ex2:<scenario>:<key>", so the answers are not readable here
    "s1": "25b3b4fe4e44f527cce8fc7f1f1822df296441014584a15369fd26636f03af78",
    "s2": "80b79027be9d1bf38bf136ba2d7052006723da2c4cd39f51acc3df217ec65e74",
    "s3": "19512965177a388ece6ff332e0c569ccd76fd66245f5acbb1a8702c4ec00ec62",
    "s4": "5b50cd473d7822521cc0176bd631f82a082c4c12fce1b35ebdfa07b72d1bd656",
    "s5": "923098e5ecb00e18c9558e6b0875a70d7eb15b654f438ee166cb39ece6ddd58d",
    "s6": "2f022a475780951dc595343614faa728a0ae4c94c78a4a0ea50656b169686ef0",
    "s7": "22c7b4f7683fae68f9e56aa96456a4047cbc720aec4061957baa1d6a7c6b6d9c",
    "s8": "534c0a862a820aae560295e9e863d898705d92e2d4ecab05544847791deff44b",
}


def check_ex2() -> bool:
    try:
        assert isinstance(PLACEMENT, dict), "PLACEMENT must be a dict"
        unknown = {s: v for s, v in PLACEMENT.items() if v not in OPTIONS}
        assert not unknown, f"use keys from OPTIONS; unknown or empty: {sorted(unknown)}"
        wrong = [s for s in _EX2_HINTS
                 if hashlib.sha256(f"ex2:{s}:{PLACEMENT.get(s)}".encode()).hexdigest() != _EX2_EXPECTED[s]]
        assert not wrong, "re-check: " + "; ".join(f"{s} ({_EX2_HINTS[s]})" for s in wrong)
    except AssertionError as e:
        print(f"✗ Exercise 2: {e}")
        RESULTS["ex2"] = False
        return False
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 2: {type(e).__name__}: {e}")
        RESULTS["ex2"] = False
        return False
    print("✓ Exercise 2 passed")
    RESULTS["ex2"] = True
    return True


check_ex2()

### Exercise 3 — Does this path-scoped rule apply? (concept 5.3)

Implement `rule_applies(paths_value, rel_path)`. It returns `True` when a rule with that `paths:` frontmatter value would load for a file at `rel_path` (a POSIX path relative to the project root). Support:

- `paths_value` as a **YAML list** of patterns, or **one comma-separated string**. Commas inside `{...}` braces do not split.
- `**` matches zero or more directories; `*` matches within one path segment (never `/`); `?` matches one character other than `/`
- brace expansion: `src/**/*.{ts,tsx}` means `src/**/*.ts` or `src/**/*.tsx`
- an empty or missing value (`None`, `""`, `[]`) matches nothing (that would be an unscoped rule, which is not a path match)

In [ ]:
def rule_applies(paths_value, rel_path: str) -> bool:
    # TODO: implement glob matching as described above
    raise NotImplementedError

In [ ]:
_EX3_CASES = [
    (["**/*.test.ts"], "src/a/b.test.ts", True),
    (["**/*.test.ts"], "b.test.ts", True),
    (["**/*.test.ts"], "src/b.test.tsx", False),
    ("src/**/*.{ts,tsx}", "src/ui/Button.tsx", True),
    ("src/**/*.{ts,tsx}", "src/index.ts", True),
    ("src/**/*.{ts,tsx}", "lib/index.ts", False),
    ("services/*/db/migrations/**", "services/billing/db/migrations/001_init.sql", True),
    ("services/*/db/migrations/**", "services/billing/api/db/migrations/x.sql", False),
    ("terraform/**/*.tf, infra/**/*.tf", "infra/network/main.tf", True),
    ("terraform/**/*.tf, infra/**/*.tf", "modules/main.tf", False),
    ("*.md", "README.md", True),
    ("*.md", "docs/guide.md", False),
    ("src/components/*.tsx", "src/components/deep/X.tsx", False),
    ("data/file?.csv", "data/file1.csv", True),
    ("data/file?.csv", "data/file10.csv", False),
    (None, "anything.py", False),
    ([], "anything.py", False),
]


def check_ex3() -> bool:
    try:
        failures = []
        for paths_value, path, expected in _EX3_CASES:
            got = rule_applies(paths_value, path)
            if got is not expected:
                failures.append(f"rule_applies({paths_value!r}, {path!r}) -> {got!r}, expected {expected}")
        assert not failures, f"{len(failures)} case(s) failed, first: {failures[0]}"
    except NotImplementedError:
        print("✗ Exercise 3: not implemented yet")
        RESULTS["ex3"] = False
        return False
    except AssertionError as e:
        print(f"✗ Exercise 3: {e}")
        RESULTS["ex3"] = False
        return False
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 3: {type(e).__name__}: {e}")
        RESULTS["ex3"] = False
        return False
    print("✓ Exercise 3 passed")
    RESULTS["ex3"] = True
    return True


check_ex3()

### Exercise 4 — Build the CI argv (concept 5.9)

A blocking PR-review job runs Claude Code from Python with `subprocess.run(argv, input=diff, ...)`. Implement `build_ci_argv(prompt, schema, model)` so that the returned **list of strings**:

- starts with `["claude", "-p", prompt]`. The prompt goes right after `-p`, because flags such as `--allowedTools` accept several values and would swallow a trailing prompt.
- prints one JSON result object, validated against `schema` (pass the schema as a JSON string)
- is reproducible on every runner: skips auto-discovery of hooks, MCP servers and CLAUDE.md, **and** still gives Claude the project's `CLAUDE.md` by appending it to the system prompt from a file
- pre-approves exactly `Read`, `Grep` and `Glob`, as **one** comma-separated value
- denies anything else that would need a permission prompt (locked-down CI mode)
- caps the run at 6 agentic turns and $2.00
- pins the model to `model`

Do not use `--dangerously-skip-permissions`.

In [ ]:
def build_ci_argv(prompt: str, schema: dict, model: str) -> list[str]:
    # TODO: return the argv list
    raise NotImplementedError

In [ ]:
_VALUE_FLAGS = {"--output-format", "--json-schema", "--append-system-prompt-file", "--allowedTools",
                "--allowed-tools", "--permission-mode", "--max-turns", "--max-budget-usd", "--model"}
_BOOL_FLAGS = {"--bare", "--no-session-persistence", "--verbose"}


def _parse_argv(argv: list[str]) -> tuple[dict, set]:
    values, bools, i = {}, set(), 3
    while i < len(argv):
        flag = argv[i]
        if flag in _BOOL_FLAGS:
            bools.add(flag)
            i += 1
        elif flag in _VALUE_FLAGS:
            assert i + 1 < len(argv), f"{flag} is missing its value"
            values[flag.replace("--allowed-tools", "--allowedTools")] = argv[i + 1]
            i += 2
        else:
            raise AssertionError(f"unexpected argument {flag!r} (only the flags this job needs)")
    return values, bools


def check_ex4() -> bool:
    schema = {"type": "object", "properties": {"findings": {"type": "array"}}, "required": ["findings"]}
    try:
        argv = build_ci_argv("Review the diff on stdin.", schema, "claude-sonnet-5-5")
        assert isinstance(argv, list) and all(isinstance(a, str) for a in argv), "return a list of str (no shell string)"
        assert argv[:3] == ["claude", "-p", "Review the diff on stdin."], "start with ['claude', '-p', prompt]"
        assert "--dangerously-skip-permissions" not in argv, "never bypass permissions in CI"
        v, b = _parse_argv(argv)
        assert v.get("--output-format") == "json", "one JSON result object: --output-format json"
        assert "--json-schema" in v and json.loads(v["--json-schema"]) == schema, "pass the schema with --json-schema '<json>'"
        assert "--bare" in b, "reproducible runs skip auto-discovery: which flag?"
        assert v.get("--append-system-prompt-file") == "CLAUDE.md", "--bare skips CLAUDE.md: append it from the file"
        tools = v.get("--allowedTools", "")
        assert sorted(t.strip() for t in tools.split(",")) == ["Glob", "Grep", "Read"], "--allowedTools 'Read,Grep,Glob' as one value"
        assert v.get("--permission-mode") == "dontAsk", "which permission mode denies anything that would prompt?"
        assert v.get("--max-turns") == "6", "--max-turns 6"
        assert float(v.get("--max-budget-usd", "nan")) == 2.0, "--max-budget-usd 2.00"
        assert v.get("--model") == "claude-sonnet-5-5", "pin the model with --model"
    except NotImplementedError:
        print("✗ Exercise 4: not implemented yet")
        RESULTS["ex4"] = False
        return False
    except AssertionError as e:
        print(f"✗ Exercise 4: {e}")
        RESULTS["ex4"] = False
        return False
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 4: {type(e).__name__}: {e}")
        RESULTS["ex4"] = False
        return False
    print("✓ Exercise 4 passed")
    RESULTS["ex4"] = True
    return True


check_ex4()

### Exercise 5 — Parse the headless JSON output (concept 5.9)

`claude -p ... --output-format json --json-schema ...` prints one JSON object. Implement `summarize_run(stdout)`, which returns a dict with exactly these keys: `ok`, `session_id`, `cost_usd`, `findings`, `error`.

| Situation | `ok` | `error` | `findings` |
|---|---|---|---|
| stdout is not valid JSON | `False` | `"invalid_json"` | `[]` (and `session_id=None`, `cost_usd=0.0`) |
| `is_error` is true **or** `subtype` is not `"success"` | `False` | the `subtype` value | `[]` |
| success, but `structured_output` is missing or has no `findings` list | `False` | `"missing_structured_output"` | `[]` |
| success with findings | `True` | `None` | the findings, sorted by severity (`critical` > `high` > `medium` > `low`), then `file`, then `line` |

Whenever the JSON parses, `session_id` comes from the output and `cost_usd` from `total_cost_usd` as a float (`0.0` if absent).

In [ ]:
SAMPLE_OK = json.dumps({
    "type": "result", "subtype": "success", "is_error": False, "num_turns": 3,
    "result": "", "session_id": "7f1c2b9e-0d4a-4c55-9a51-2f3e8d6b1a00", "total_cost_usd": 0.0412,
    "usage": {"input_tokens": 9120, "output_tokens": 640},
    "structured_output": {"findings": [
        {"file": "src/pricing.py", "line": 4, "severity": "high", "issue": "rate may be None", "suggested_fix": "default 0.0"},
        {"file": "src/api.py", "line": 12, "severity": "low", "issue": "unused import", "suggested_fix": "remove it"},
        {"file": "src/pricing.py", "line": 5, "severity": "critical", "issue": "multiplies by rate", "suggested_fix": "1 - rate"},
        {"file": "src/api.py", "line": 3, "severity": "high", "issue": "no auth check", "suggested_fix": "add guard"},
    ]},
})
SAMPLE_MAX_TURNS = json.dumps({
    "type": "result", "subtype": "error_max_turns", "is_error": True, "num_turns": 6,
    "session_id": "0b9d6c1e-77aa-4f0e-b2f1-6d0c3a9e5b11", "total_cost_usd": 0.3,
})
SAMPLE_NO_SCHEMA = json.dumps({
    "type": "result", "subtype": "success", "is_error": False, "num_turns": 1,
    "result": "Looks fine to me!", "session_id": "c3d2e1f0-1111-4222-8333-444455556666", "total_cost_usd": 0.01,
})


def summarize_run(stdout: str) -> dict:
    # TODO: implement the table above
    raise NotImplementedError

In [ ]:
def check_ex5() -> bool:
    try:
        r = summarize_run(SAMPLE_OK)
        assert set(r) == {"ok", "session_id", "cost_usd", "findings", "error"}, f"keys must be exactly ok/session_id/cost_usd/findings/error, got {sorted(r)}"
        assert r["ok"] is True and r["error"] is None, "SAMPLE_OK should be ok=True, error=None"
        assert r["session_id"] == "7f1c2b9e-0d4a-4c55-9a51-2f3e8d6b1a00", "session_id comes from the JSON"
        assert isinstance(r["cost_usd"], float) and abs(r["cost_usd"] - 0.0412) < 1e-9, "cost_usd from total_cost_usd"
        order = [(f["severity"], f["file"], f["line"]) for f in r["findings"]]
        assert order == [("critical", "src/pricing.py", 5), ("high", "src/api.py", 3),
                         ("high", "src/pricing.py", 4), ("low", "src/api.py", 12)], f"sort by severity, file, line; got {order}"

        r = summarize_run(SAMPLE_MAX_TURNS)
        assert r["ok"] is False and r["error"] == "error_max_turns" and r["findings"] == [], "errors: ok=False, error=subtype"
        assert r["session_id"] == "0b9d6c1e-77aa-4f0e-b2f1-6d0c3a9e5b11" and r["cost_usd"] == 0.3, "still report session_id and cost on errors"

        r = summarize_run(SAMPLE_NO_SCHEMA)
        assert r["ok"] is False and r["error"] == "missing_structured_output", "prose without structured_output is not ok"

        r = summarize_run("Error: something went wrong")
        assert r == {"ok": False, "session_id": None, "cost_usd": 0.0, "findings": [], "error": "invalid_json"}, f"invalid JSON case, got {r}"
    except NotImplementedError:
        print("✗ Exercise 5: not implemented yet")
        RESULTS["ex5"] = False
        return False
    except AssertionError as e:
        print(f"✗ Exercise 5: {e}")
        RESULTS["ex5"] = False
        return False
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 5: {type(e).__name__}: {e}")
        RESULTS["ex5"] = False
        return False
    print("✓ Exercise 5 passed")
    RESULTS["ex5"] = True
    return True


check_ex5()

### Exercise 6 — Plan the refinement messages (concept 5.11)

The test suite of an order-import tool has five failing tests (`FAILING` below). You will hand them to Claude Code. Implement `plan_fix_messages(failures)`, which returns a **list of message strings**, one per message you would send:

- Issues that **interact** go into **one** message. Two issues interact if either lists the other in `interacts_with`, and interaction chains (if A interacts with B and B with C, all three share a message).
- Every **independent** issue gets **its own** message.
- Order the messages by the first failing test they contain (the order of `FAILING`), and keep that order inside each message too.
- For each issue, its message must include the **test name**, the **input** and **expected** values written with `repr()`, and the **actual** output string, so Claude gets an exact target instead of "it's still broken".

In [ ]:
FAILING = [
    {"id": "null-email", "test": "test_migrate_null_email", "input": {"email": None}, "expected": {"email": None},
     "actual": "AttributeError: 'NoneType' object has no attribute 'lower'", "interacts_with": []},
    {"id": "tz-dropped", "test": "test_parse_keeps_tz", "input": "2026-03-01T09:00:00+02:00",
     "expected": "2026-03-01T07:00:00Z", "actual": "'2026-03-01T09:00:00Z'", "interacts_with": ["dedupe-ts"]},
    {"id": "help-typo", "test": "test_help_text", "input": ["--help"], "expected": "Usage: import [OPTIONS]",
     "actual": "'Usgae: import [OPTIONS]'", "interacts_with": []},
    {"id": "dedupe-ts", "test": "test_dedupe_same_instant",
     "input": ["2026-03-01T09:00:00+02:00", "2026-03-01T07:00:00Z"], "expected": 1, "actual": "2", "interacts_with": []},
    {"id": "dedupe-window", "test": "test_dedupe_window_minutes", "input": {"window_minutes": 5}, "expected": 3,
     "actual": "4", "interacts_with": ["dedupe-ts"]},
]


def plan_fix_messages(failures: list[dict]) -> list[str]:
    # TODO: group interacting issues, then write one message per group
    raise NotImplementedError

In [ ]:
def check_ex6() -> bool:
    expected_groups = [["test_migrate_null_email"],
                       ["test_parse_keeps_tz", "test_dedupe_same_instant", "test_dedupe_window_minutes"],
                       ["test_help_text"]]
    try:
        msgs = plan_fix_messages(FAILING)
        assert isinstance(msgs, list) and all(isinstance(m, str) for m in msgs), "return a list of message strings"
        by_test = {f["test"]: f for f in FAILING}
        got_groups = [[t for t in by_test if t in m] for m in msgs]
        if len(msgs) == len(FAILING):
            raise AssertionError("one message per issue: the interacting timestamp/dedupe issues belong in ONE message")
        if len(msgs) == 1:
            raise AssertionError("everything in one message: independent issues (null email, help typo) get their own")
        sets = [set(g) for g in got_groups]
        assert sets == [set(g) for g in expected_groups], (
            f"wrong grouping or order: {got_groups}. Remember that interaction is two-way and chains")
        for m, group in zip(msgs, expected_groups):
            positions = [m.index(t) for t in group]
            assert positions == sorted(positions), "keep the FAILING order inside a message"
            for t in group:
                f = by_test[t]
                for label, needle in (("input", repr(f["input"])), ("expected", repr(f["expected"])),
                                      ("actual", f["actual"])):
                    assert needle in m, f"{t}: include the {label} value {needle}"
    except NotImplementedError:
        print("✗ Exercise 6: not implemented yet")
        RESULTS["ex6"] = False
        return False
    except AssertionError as e:
        print(f"✗ Exercise 6: {e}")
        RESULTS["ex6"] = False
        return False
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 6: {type(e).__name__}: {e}")
        RESULTS["ex6"] = False
        return False
    print("✓ Exercise 6 passed")
    RESULTS["ex6"] = True
    return True


check_ex6()

### Exercise 7 — Claude Code or Computer Use? (concept 5.12)

Implement `choose_automation(job)`, which returns the right Anthropic app (or no app) for an automation job, as one of these strings:

| Return value | Meaning |
|---|---|
| `"claude_code_headless"` | Claude Code with `claude -p` (CI, cron, GitHub Actions) |
| `"claude_code_interactive"` | A person runs a normal Claude Code session |
| `"claude_code_computer_use"` | Claude Code's built-in computer use drives the person's own desktop |
| `"computer_use_api"` | Your own app on the `computer_toolset_20260801` API toolset, in a sandbox |
| `"api_or_mcp"` | Not screen control at all: call the system's API, CLI or MCP server as a tool |

Each `job` is a dict with: `kind` (`"code"` = change or review code, `"app"` = operate an application), `has_api` (the application has an API, CLI or MCP server), `unattended` (no person present), `auth` (`"claude.ai"`, `"bedrock"`, `"google-cloud"` or `"foundry"`), `plan` (`"pro"`, `"max"`, `"team"` or `"enterprise"`), `surface` (`"cli"` or `"desktop"`) and `os` (`"macos"`, `"windows"` or `"linux"`).

Rules, from the README:

- Code work goes to Claude Code: headless when unattended, otherwise interactive.
- For operating an app, prefer its API, CLI or MCP server whenever one exists.
- Claude Code's computer use requires an interactive session, a Pro or Max plan, a claude.ai login, and macOS for the CLI (macOS or Windows for the Desktop app). When any of these fails, the job goes to the API toolset.

In [ ]:
def choose_automation(job: dict) -> str:
    # TODO: return one of the five strings in the table above
    raise NotImplementedError

In [ ]:
EX7_CASES = [
    ({"kind": "code", "has_api": False, "unattended": True, "auth": "bedrock", "plan": "enterprise",
      "surface": "cli", "os": "linux"}, "claude_code_headless", "nightly PR review in CI is code work, run headless"),
    ({"kind": "code", "has_api": False, "unattended": False, "auth": "claude.ai", "plan": "pro",
      "surface": "cli", "os": "macos"}, "claude_code_interactive", "a developer refactoring code at their desk"),
    ({"kind": "app", "has_api": True, "unattended": True, "auth": "claude.ai", "plan": "max",
      "surface": "cli", "os": "macos"}, "api_or_mcp", "the CRM has a REST API, so screen control is the wrong tool"),
    ({"kind": "app", "has_api": False, "unattended": False, "auth": "claude.ai", "plan": "max",
      "surface": "cli", "os": "macos"}, "claude_code_computer_use", "click through a native app on your own Mac"),
    ({"kind": "app", "has_api": False, "unattended": False, "auth": "claude.ai", "plan": "pro",
      "surface": "desktop", "os": "windows"}, "claude_code_computer_use", "the Desktop app offers computer use on Windows"),
    ({"kind": "app", "has_api": False, "unattended": False, "auth": "claude.ai", "plan": "pro",
      "surface": "cli", "os": "windows"}, "computer_use_api", "CLI computer use is macOS only"),
    ({"kind": "app", "has_api": False, "unattended": True, "auth": "claude.ai", "plan": "max",
      "surface": "cli", "os": "macos"}, "computer_use_api", "unattended means -p, where Claude Code's computer use is not available"),
    ({"kind": "app", "has_api": False, "unattended": False, "auth": "google-cloud", "plan": "enterprise",
      "surface": "cli", "os": "macos"}, "computer_use_api", "cloud-provider auth and Enterprise plans can't use Claude Code's computer use"),
    ({"kind": "app", "has_api": False, "unattended": False, "auth": "claude.ai", "plan": "team",
      "surface": "desktop", "os": "macos"}, "computer_use_api", "the research preview is for Pro and Max plans, not Team"),
]


def check_ex7() -> bool:
    try:
        for job, want, why in EX7_CASES:
            got = choose_automation(dict(job))
            assert got == want, f"{why}: expected {want!r}, got {got!r}"
    except NotImplementedError:
        print("✗ Exercise 7: not implemented yet")
        RESULTS["ex7"] = False
        return False
    except AssertionError as e:
        print(f"✗ Exercise 7: {e}")
        RESULTS["ex7"] = False
        return False
    except Exception as e:  # noqa: BLE001
        print(f"✗ Exercise 7: {type(e).__name__}: {e}")
        RESULTS["ex7"] = False
        return False
    print("✓ Exercise 7 passed")
    RESULTS["ex7"] = True
    return True


check_ex7()

## Part C — Architecture scenario

**Acme Retail** has 60 engineers and one monorepo: three Python services under `services/`, a React frontend in `web/`, and Terraform under `infra/` and `services/*/infra/`. They are rolling out Claude Code and ask you for a configuration design that meets these requirements:

1. Every engineer gets the same build, test and branch conventions from day one.
2. A security reminder ("never paste customer data or secrets into prompts") applies on every company laptop, and nobody can switch it off.
3. Terraform must tag every resource with `owner` and `cost_center`. The rule should not cost context when people work on the frontend.
4. The 35-step release runbook and the "add a new payment provider" guide matter only during those tasks.
5. A blocking PR review in GitHub Actions posts inline comments (file, line, severity, fix). It must behave the same on every runner and must not be influenced by developers' personal configs.
6. Production databases must never be dropped by Claude, even if someone writes a careless prompt.
7. After a long investigation of the checkout service, engineers want to compare two refactoring strategies in code.

Write your design below: where each piece lives, which flags and frontmatter you use, and the trade-offs. Aim for 150–300 words.

*Your answer here.*

<details>
<summary>Rubric: what a strong answer contains (open after writing yours)</summary>

- **(1)** Project `CLAUDE.md` (or `.claude/CLAUDE.md`), short (under ~200 lines), reviewed through PRs. Not `~/.claude/CLAUDE.md`, which new hires would not get.
- **(2)** Managed-policy CLAUDE.md deployed by IT/MDM (macOS `/Library/Application Support/ClaudeCode/CLAUDE.md`, Linux `/etc/claude-code/CLAUDE.md`), which cannot be excluded. A strong answer notes that it is still *guidance*, and pairs it with managed settings for anything that must be enforced.
- **(3)** `.claude/rules/terraform.md` with `paths: "infra/**/*.tf, services/*/infra/**/*.tf"`. It loads only when a matching file is read, written or edited, so frontend work pays nothing. Not a CLAUDE.md per directory, and not `@imports` (which load at launch).
- **(4)** Skills in `.claude/skills/<name>/SKILL.md` with precise `description` + "Use when" text. The release skill gets `disable-model-invocation: true` (side effects); supporting files hold the long checklists. CLAUDE.md stays lean.
- **(5)** `claude -p` with `--output-format json --json-schema` → `structured_output` → GitHub inline-comment API; `--bare` plus `--append-system-prompt-file CLAUDE.md` for reproducibility; `--allowedTools` read-only list, `--permission-mode dontAsk`, `--max-turns`, `--max-budget-usd`, job timeout; key from a repo/org secret; an **independent** reviewer session; re-reviews pass the previous findings. Either `claude-code-action@v1` with `claude_args` or a custom script is acceptable if justified.
- **(6)** A `permissions.deny` rule (for example on the DB CLI's drop commands) or a `PreToolUse` hook, in managed or project settings. **Not** a CLAUDE.md sentence.
- **(7)** Fork the investigation session (`--resume <id> --fork-session` or `/branch`), with each fork in its own git worktree (`claude -w`), because forks share the filesystem. If the code changed a lot since the investigation, start fresh sessions seeded with a structured summary instead.
- Trade-offs mentioned: context cost vs reliability, guidance vs enforcement, auto-invocation vs human control.

</details>

## Scorecard

In [ ]:
def scorecard(pass_mark: float = 72.0) -> None:
    quiz_score, quiz_total = RESULTS.get("quiz", (0, 12))
    exercises = [RESULTS.get(f"ex{i}", False) for i in range(1, 8)]
    quiz_pct = 100 * quiz_score / quiz_total
    ex_pct = 100 * sum(bool(x) for x in exercises) / len(exercises)
    overall = (quiz_pct + ex_pct) / 2
    print(f"Quiz:      {quiz_score}/{quiz_total} ({quiz_pct:.0f}%)")
    print(f"Exercises: {sum(bool(x) for x in exercises)}/{len(exercises)} ({ex_pct:.0f}%)  "
          + " ".join(f"ex{i}{'✓' if ok else '✗'}" for i, ok in enumerate(exercises, 1)))
    print(f"Overall:   {overall:.0f}%  ->  {'PASS' if overall >= pass_mark else 'RETRY'} (pass mark {pass_mark:.0f}%)")
    print("Part C:    self-assessed against the rubric")


scorecard()

---

## Solutions (spoilers)

Open these only after you have tried everything.

<details>
<summary>Part A — quiz answers and explanations</summary>

- **Q1: C.** `CLAUDE.local.md` is personal and project-specific; add it to `.gitignore`. A would push your port to the whole team. B applies to every project you open. D is a committed file, so it is shared, and `paths: "**/*"` only makes it load lazily.

- **Q2: B.** Imported files are expanded into context **at launch**, next to the importing file. Imports organize; they do not save tokens. Use path-scoped rules (conditional) or skills (on demand) to cut context cost. There is no three-import limit (nesting is capped at four hops), and imports can point almost anywhere, including `~/...`, with an approval dialog for external paths.

- **Q3: D.** Path-scoped rules trigger on **Read, Write or Edit** of a matching file. Reading only markdown files does not match `**/*.sql`. Unscoped rules (no `paths`) are the ones that load at launch. `/rules` is not a command.

- **Q4: A.** `disable-model-invocation: true` removes the skill from Claude's listing, so only a human can run `/deploy-prod`. B is the opposite (Claude-only, hidden from the `/` menu). C relies on wording the model may ignore. D only pre-approves a tool; it does not stop invocation.

- **Q5: D.** Per current docs, `disallowed-tools` removes tools from Claude's pool while the skill is active. `allowed-tools` only **pre-approves** the listed tools; others remain callable through normal permissions. `context: fork` isolates context, but the default subagent still has tools. A description is not a control.

- **Q6: C.** `--bare` skips auto-discovery of hooks, skills, plugins, MCP servers, auto memory and CLAUDE.md, so every runner behaves the same. Pass what the job needs explicitly (`--append-system-prompt-file`, `--mcp-config`, `--settings`). Bare mode authenticates only with `ANTHROPIC_API_KEY` or an `apiKeyHelper`.

- **Q7: A.** Many files, several viable designs and unfamiliar code are exactly the plan-mode case: explore, propose, get approval, then execute. B assumes you already know the structure. C reacts to problems instead of preventing them. D does nothing for design risk.

- **Q8: D.** Sessions created with `-p` or the SDK are left out of the `/resume` picker and out of interactive `claude --continue`, but they are saved. Capture `session_id` from the JSON output and resume by ID (`claude -p --continue` would include them).

- **Q9: B.** Forking gives each strategy the shared history with its own session ID. A fork copies the conversation, **not the filesystem**, so forks that edit code need separate git worktrees (`claude -w <name>`). Resuming one session in two terminals interleaves both into one transcript.

- **Q10: C.** After compaction, the project-root CLAUDE.md (and unscoped rules, auto memory, the plan, invoked skill bodies) are re-injected from disk. Path-scoped rules and nested CLAUDE.md files were in message history, so they are summarized away until a matching file is read again. Chat-only instructions survive only if the summary kept them.

- **Q11: A.** When prose keeps being read differently, concrete input/output examples remove the ambiguity, and turning them into tests gives Claude a pass/fail signal to iterate against. B and C repeat the same ambiguous prose. D throws away useful history and changes nothing about the instruction.

- **Q12: D.** `stream-json` emits one JSON event per line while the run is in progress. With `--include-partial-messages` the text arrives as `stream_event` lines carrying `content_block_delta` / `text_delta`, and the last line is the `result` message with cost and session ID. Pair it with `--verbose`, as every documented example does. A prints nothing until the end. B is interactive and not machine-readable. C cannot be parsed reliably and has no metadata.

</details>

<details>
<summary>Exercise 1 — SKILL.md</summary>

```python
def write_db_migrate_skill() -> str:
    return """---
name: db-migrate
description: Create a new database migration file that follows the team conventions. Use when the user asks to create, add or scaffold a migration.
argument-hint: "[migration-name]"
disable-model-invocation: true
context: fork
agent: general-purpose
allowed-tools: Bash(cat db/VERSION) Read Write
disallowed-tools: WebFetch WebSearch
---
Create a migration named `$ARGUMENTS` in `db/migrations/`.
Current schema version: !`cat db/VERSION`
Read `db/MIGRATIONS.md` first and follow its conventions. Never modify existing migrations.
"""
```

`disallowed-tools` removes tools; `allowed-tools` only pre-approves them (and is needed so the `` !`cat db/VERSION` `` injection is not aborted). `agent` is optional; it picks the subagent type for the fork.

</details>

<details>
<summary>Exercise 2 — placement</summary>

```python
PLACEMENT = {
    "s1": "project",      # versioned, arrives with git clone
    "s2": "user",         # personal, every project
    "s3": "managed",      # org-wide, cannot be excluded
    "s4": "path_rule",    # file type scattered across directories
    "s5": "skill",        # task-specific, loads on demand
    "s6": "permissions",  # must be enforced, not suggested
    "s7": "subdir",       # one self-contained directory
    "s8": "local",        # personal, this repo only, gitignored
}
```

Ask two questions: *who* must get it (everyone, only me, the whole org) and *when* it should load (always, for certain files, for a task, or enforced).

</details>

<details>
<summary>Exercise 3 — rule_applies</summary>

```python
def rule_applies(paths_value, rel_path: str) -> bool:
    if isinstance(paths_value, list):
        patterns = [str(p).strip() for p in paths_value]
    else:
        patterns = [p.strip() for p in re.split(r",(?![^{]*\})", str(paths_value or ""))]
    patterns = [p for p in patterns if p]

    def expand_braces(pattern: str) -> list[str]:
        m = re.search(r"\{([^{}]*)\}", pattern)
        if not m:
            return [pattern]
        return [x for alt in m.group(1).split(",")
                for x in expand_braces(pattern[:m.start()] + alt + pattern[m.end():])]

    def to_regex(pattern: str) -> str:
        out, i = "", 0
        while i < len(pattern):
            if pattern.startswith("**/", i):
                out, i = out + "(?:.*/)?", i + 3
            elif pattern.startswith("**", i):
                out, i = out + ".*", i + 2
            elif pattern[i] == "*":
                out, i = out + "[^/]*", i + 1
            elif pattern[i] == "?":
                out, i = out + "[^/]", i + 1
            else:
                out, i = out + re.escape(pattern[i]), i + 1
        return out + r"\Z"

    return any(re.match(to_regex(p), rel_path) for pat in patterns for p in expand_braces(pat))
```

`**/` becomes an optional run of directories, so `**/*.test.ts` also matches a file at the root. Split comma-separated strings only on commas outside braces.

</details>

<details>
<summary>Exercise 4 — build_ci_argv</summary>

```python
def build_ci_argv(prompt: str, schema: dict, model: str) -> list[str]:
    return [
        "claude", "-p", prompt,
        "--output-format", "json",
        "--json-schema", json.dumps(schema),
        "--bare",
        "--append-system-prompt-file", "CLAUDE.md",
        "--allowedTools", "Read,Grep,Glob",
        "--permission-mode", "dontAsk",
        "--max-turns", "6",
        "--max-budget-usd", "2.00",
        "--model", model,
    ]
```

`--bare` skips CLAUDE.md, so the project file comes back through `--append-system-prompt-file`. `dontAsk` denies anything not pre-approved. Values are strings because argv is a list of strings.

</details>

<details>
<summary>Exercise 5 — summarize_run</summary>

```python
def summarize_run(stdout: str) -> dict:
    rank = {"critical": 0, "high": 1, "medium": 2, "low": 3}
    try:
        out = json.loads(stdout)
    except (json.JSONDecodeError, TypeError):
        return {"ok": False, "session_id": None, "cost_usd": 0.0, "findings": [], "error": "invalid_json"}
    base = {"session_id": out.get("session_id"), "cost_usd": float(out.get("total_cost_usd") or 0.0)}
    if out.get("is_error") or out.get("subtype") != "success":
        return {"ok": False, **base, "findings": [], "error": out.get("subtype")}
    findings = (out.get("structured_output") or {}).get("findings")
    if not isinstance(findings, list):
        return {"ok": False, **base, "findings": [], "error": "missing_structured_output"}
    findings = sorted(findings, key=lambda f: (rank.get(f.get("severity"), 99), f.get("file", ""), f.get("line", 0)))
    return {"ok": True, **base, "findings": findings, "error": None}
```

Do not trust the exit code alone: a run that hits `--max-turns` still prints a result object with `is_error: true` and `subtype: "error_max_turns"`.

</details>

<details>
<summary>Exercise 6 — plan_fix_messages</summary>

```python
def plan_fix_messages(failures: list[dict]) -> list[str]:
    parent = {f["id"]: f["id"] for f in failures}

    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x

    for f in failures:                      # undirected links, so chains merge too
        for other in f["interacts_with"]:
            parent[find(f["id"])] = find(other)

    groups: dict[str, list[dict]] = {}
    for f in failures:                      # dict keeps first-appearance order
        groups.setdefault(find(f["id"]), []).append(f)

    messages = []
    for group in groups.values():
        head = ("These failing tests interact, so fix them together in one consistent change:"
                if len(group) > 1 else "Fix this failing test:")
        lines = [head] + [f"- {f['test']}: input {f['input']!r}, expected {f['expected']!r}, got {f['actual']}"
                          for f in group]
        lines.append("Run the tests afterwards and don't edit them.")
        messages.append("\n".join(lines))
    return messages
```

Interacting issues share one message so Claude designs one consistent fix; independent ones stay small and separately reviewable. Every line names the test and gives input, expected and actual values, which is the test-driven feedback loop from 5.11.

</details>

<details>
<summary>Exercise 7 — choose_automation</summary>

```python
def choose_automation(job: dict) -> str:
    if job["kind"] == "code":
        return "claude_code_headless" if job["unattended"] else "claude_code_interactive"
    if job["has_api"]:
        return "api_or_mcp"                      # screen control is the broadest, slowest option
    supported_os = {"cli": {"macos"}, "desktop": {"macos", "windows"}}
    claude_code_ok = (not job["unattended"]                 # no -p
                      and job["plan"] in {"pro", "max"}     # research preview
                      and job["auth"] == "claude.ai"        # not Bedrock / Google Cloud / Foundry
                      and job["os"] in supported_os.get(job["surface"], set()))
    return "claude_code_computer_use" if claude_code_ok else "computer_use_api"
```

Claude Code automates development work; Computer Use automates GUIs. An API or MCP server beats reading pixels. Claude Code's computer use is an interactive research preview on your own desktop, so unattended or cloud-provider jobs go to the API toolset, hosted in a sandbox with a domain allowlist and human confirmation for consequential actions.

</details>

---

**Next chapter:** [Chapter 6 — Prompt Engineering](../06-prompt-engineering/README.md)